# Hyperparameter tuning analysis

How an Optuna study from `src/hyperparameter_tuning.py` went. Each trial trains the multiclass BDT with one
set of XGBoost hyperparameters (fixed learning rate), early-stopped on the validation split, and its objective
is the best validation log loss (with the study's event weights: unweighted for weight mode `unit`). Trials are
proposed by a multivariate TPE sampler, and a median pruner stops trials whose validation loss is above the
median of the completed trials at the same boosting round. The first trial is the XGBoost defaults.

Each trial also records the validation log loss with the physics (POT) weights, raw and after moving the class
priors to the physics priors (`val_physics_weighted_logloss`), the loss that matters for the analysis prediction.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

study_name = "hp_2026_10_25"

tuning_dir = "../hyperparameter_tuning_outputs"
plot_dir = f"../plots/hyperparameter_tuning/{study_name}"
os.makedirs(plot_dir, exist_ok=True)


def load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{tuning_dir}/{name}.db")


def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{plot_dir}/{name}.png", dpi=150)
    plt.show()

# Overview

In [ ]:
study = load_study(study_name)
setup = study.user_attrs.get("setup", {})
complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
distributions = complete[0].distributions  # the tuned parameters and their search ranges
param_names = list(distributions)

trials = study.trials_dataframe(attrs=("number", "value", "duration", "params", "user_attrs", "state"))
trials.columns = [c.removeprefix("params_").removeprefix("user_attrs_") for c in trials.columns]
trials["minutes"] = trials["duration"].dt.total_seconds() / 60
trials["last_round"] = [max(t.intermediate_values, default=np.nan) for t in study.trials]
trials["last_loss"] = [t.intermediate_values[max(t.intermediate_values)] if t.intermediate_values else np.nan for t in study.trials]
done = trials[trials["state"] == "COMPLETE"]

best, default = study.best_trial, study.trials[0]
print("setup:", {k: v for k, v in setup.items() if k not in ("input_file", "training_vars_sha1")})
print(f"trials: {len(complete)} complete, {len(pruned)} pruned, {trials['minutes'].sum() / 60:.1f} hours total")
print(f"default params (trial 0): validation loss {default.value:.5f}, "
      f"physics-weighted {default.user_attrs['val_physics_weighted_logloss']:.5f}, best round {default.user_attrs['best_iteration']}")
print(f"best (trial {best.number}):        validation loss {best.value:.5f}, "
      f"physics-weighted {best.user_attrs['val_physics_weighted_logloss']:.5f}, best round {best.user_attrs['best_iteration']}")
print(f"improvement over the defaults: {default.value - best.value:.5f} ({(default.value - best.value) / default.value:.2%}) "
      f"in validation loss, {default.user_attrs['val_physics_weighted_logloss'] - best.user_attrs['val_physics_weighted_logloss']:.5f} "
      f"in physics-weighted loss")

top_columns = ["number", "value", "val_physics_weighted_logloss", "best_iteration", "minutes"] + param_names
with pd.option_context("display.float_format", "{:.4g}".format, "display.width", 250):
    display(done.sort_values("value")[top_columns].head(10).reset_index(drop=True))

# Optimization history

Completed trials at their final validation loss, pruned trials at the loss where they were stopped (or at the
top edge, if that is above the plotted range), and the best value so far. A flattening best-so-far line means more trials are unlikely to help much.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pr = trials[trials["state"] == "PRUNED"]
y_low, y_high = best.value - 0.005, np.quantile(done["value"], 0.9) + 0.01
ax.scatter(done["number"], done["value"], color="tab:blue", label="complete")
ax.scatter(pr["number"], np.minimum(pr["last_loss"], y_high - 0.001), color="gray", marker="x",
           label=f"pruned ({len(pr)}; loss when stopped, capped at the top edge)")
best_so_far = done.sort_values("number").set_index("number")["value"].cummin()
ax.step(best_so_far.index, best_so_far.values, where="post", color="tab:red", label="best so far")
ax.scatter([0], [default.value], marker="*", s=300, color="tab:orange", edgecolors="black", zorder=3, label="XGBoost defaults")
ax.set_ylim(y_low, y_high)
ax.set_xlabel("Trial")
ax.set_ylabel("Validation log loss")
ax.set_title(f"{study_name}: optimization history")
ax.legend()
save(fig, "optimization_history")

# Learning curves and pruning

Validation loss vs boosting round for every trial (complete trials colored by their final loss, pruned trials
in gray). The zoomed panel shows how close the good trials are, and where the pruner stopped the others.

In [ ]:
norm = mpl.colors.Normalize(done["value"].min(), np.quantile(done["value"], 0.9))
cmap = plt.cm.viridis
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
for ax, zoom in zip(axs, [False, True]):
    for t in study.trials:
        steps = sorted(t.intermediate_values)
        losses = [t.intermediate_values[s] for s in steps]
        if t.state == optuna.trial.TrialState.COMPLETE:
            ax.plot(steps, losses, color=cmap(norm(t.value)), lw=1.5 if t.number == best.number else 0.8,
                    zorder=3 if t.number == best.number else 2)
        else:
            ax.plot(steps, losses, color="gray", lw=0.6, alpha=0.6, zorder=1)
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("Validation log loss")
    if zoom:
        ax.set_ylim(best.value - 0.005, best.value + 0.08)
        ax.set_title("zoomed")
    else:
        ax.set_yscale("log")
        ax.set_title(f"{study_name}: all trials")
fig.colorbar(mpl.cm.ScalarMappable(norm, cmap), ax=axs, label="final validation loss (complete trials)")
fig.savefig(f"{plot_dir}/learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Which hyperparameters mattered

fANOVA importances (the fraction of the variance of the objective explained by each hyperparameter, from a
random forest fit to the completed trials), for the validation loss and for the training time per trial.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
importances = {
    "validation log loss": optuna.importance.get_param_importances(study, params=param_names),
    "minutes per trial": optuna.importance.get_param_importances(
        study, params=param_names, target=lambda t: t.duration.total_seconds() / 60),
}
for ax, (target, imp) in zip(axs, importances.items()):
    names = list(imp)[::-1]
    ax.barh(names, [imp[n] for n in names], color="tab:blue")
    ax.set_xlabel("Importance")
    ax.set_title(f"Importance for the {target}")
save(fig, "importances")

# Validation loss vs each hyperparameter

Completed trials (colored by trial number, so later points are where the sampler concentrated), with the search
range edges as dashed lines and the best trial as a red star. A best value at a range edge suggests widening the
range.

In [ ]:
ncols = 3
nrows = -(-len(param_names) // ncols)
fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
for ax, name in zip(axs.flat, param_names):
    dist = distributions[name]
    sc = ax.scatter(done[name], done["value"], c=done["number"], cmap="plasma", s=25)
    ax.scatter(best.params[name], best.value, marker="*", s=300, color="tab:red", edgecolors="black", zorder=3)
    ax.scatter(default.params[name], default.value, marker="*", s=200, color="tab:orange", edgecolors="black", zorder=3)
    for edge in (dist.low, dist.high):
        ax.axvline(edge, color="k", ls="--", lw=1)
    if getattr(dist, "log", False):
        ax.set_xscale("log")
    ax.set_ylim(best.value - 0.003, np.quantile(done["value"], 0.9) + 0.005)
    ax.set_xlabel(name)
for ax in axs[:, 0]:
    ax.set_ylabel("Validation log loss")
for ax in axs.flat[len(param_names):]:
    ax.set_visible(False)
fig.colorbar(sc, ax=axs, label="trial number")
fig.suptitle(f"{study_name}: red star = best trial, orange star = XGBoost defaults")
fig.savefig(f"{plot_dir}/loss_vs_parameters.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def range_position(dist, value):
    # 0 at the low edge of the search range, 1 at the high edge (in log space for log-scale parameters)
    if getattr(dist, "log", False):
        return (np.log(value) - np.log(dist.low)) / (np.log(dist.high) - np.log(dist.low))
    return (value - dist.low) / (dist.high - dist.low)


top5 = done.sort_values("value").head(5)
positions = pd.DataFrame({name: [range_position(distributions[name], v) for v in top5[name]] for name in param_names},
                         index=[f"trial {n}" for n in top5["number"]])
with pd.option_context("display.float_format", "{:.2f}".format):
    print("position in the search range of the 5 best trials (0 = low edge, 1 = high edge):")
    display(positions)
for name in param_names:
    pos = positions[name].iloc[0]
    if pos < 0.05 or pos > 0.95:
        edge = "low" if pos < 0.05 else "high"
        print(f"best trial's {name} = {best.params[name]:.4g} is at the {edge} edge of [{distributions[name].low:g}, "
              f"{distributions[name].high:g}]: consider widening the range")

In [ ]:
# the two most important hyperparameters together
first, second = list(importances["validation log loss"])[:2]
fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(done[first], done[second], c=done["value"], cmap="viridis_r", s=60,
                vmin=done["value"].min(), vmax=np.quantile(done["value"], 0.9))
ax.scatter(best.params[first], best.params[second], marker="*", s=400, color="tab:red", edgecolors="black")
for name, axis in [(first, "x"), (second, "y")]:
    if getattr(distributions[name], "log", False):
        getattr(ax, f"set_{axis}scale")("log")
ax.set_xlabel(first)
ax.set_ylabel(second)
fig.colorbar(sc, label="validation log loss")
ax.set_title(f"{study_name}: the two most important hyperparameters")
save(fig, "two_most_important")

# Optimized loss vs the physics-weighted loss

The study optimizes the validation loss with its own event weights; the physics-weighted loss (after the class
prior correction) is what matters for the analysis prediction. If they rank trials the same way, the tuning
also optimized the physics-weighted performance.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axs, ["val_physics_weighted_logloss_raw", "val_physics_weighted_logloss"]):
    ax.scatter(done["value"], done[col], c=done["number"], cmap="plasma")
    ax.scatter(best.value, best.user_attrs[col], marker="*", s=300, color="tab:red", edgecolors="black")
    rho = done[["value", col]].corr(method="spearman").iloc[0, 1]
    ax.set_xlabel("Validation log loss (optimized)")
    ax.set_ylabel(col)
    ax.set_title(f"rank correlation {rho:.2f}")
save(fig, "objective_vs_physics_weighted")
best_physics = done.loc[done["val_physics_weighted_logloss"].idxmin()]
print(f"best physics-weighted loss: trial {best_physics['number']} ({best_physics['val_physics_weighted_logloss']:.5f}); "
      f"best objective: trial {best.number} ({best.user_attrs['val_physics_weighted_logloss']:.5f})")

# Cost

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
sc = axs[0].scatter(done["best_iteration"], done["minutes"], c=done["max_depth"], cmap="cividis", s=40)
fig.colorbar(sc, ax=axs[0], label="max_depth")
axs[0].set_xlabel("Best boosting round")
axs[0].set_ylabel("Minutes per trial")
axs[0].set_title("Completed trials")
axs[1].bar(["complete", "pruned"], [done["minutes"].sum() / 60, trials.loc[trials["state"] == "PRUNED", "minutes"].sum() / 60],
           color=["tab:blue", "gray"])
axs[1].set_ylabel("Hours")
axs[1].set_title("Total time")
save(fig, "cost")